# Фінальний проект "Населення"

### Виконує:

#### Онищенко Богдан \ Onyshchenko Bohdan \ vumenira


In [1]:
import pandas as pd
import altair as alt

## Частина 1: Підготовка даних

In [5]:
people_1778 = pd.read_csv("2.2_Дунаєць_1778.xlsx - База.csv")


people_1778.columns = people_1778.iloc[2].values
people_1778 = people_1778.iloc[3:].reset_index(drop = True)

people_1778.columns.values[0] = "Аркуш"
people_1778.columns.values[4] = "Чоловіки наскрізне"
people_1778.columns.values[5] = "Жінки наскрізне"

def is_male(x):                                 # Приводячи датасет до зручного виду, я розумію що втрачаю багато інформації видаляючи зайві колонки
    if str(x).replace(".", "").isdigit():       # Тут я роблю це усвідомлено, бо знаю, що не використовуватиму їх при порівнянні
        return "Чол"                            # Адже для порівняння мені не потрібні дані, які є лише в 1ому з датасетів
    else:                                       # За інших умов можна було б їх зберегти, трохи переробивши код
        return "Жін"
people_1778["Стать"] = people_1778["Чоловіки наскрізне"].apply(is_male)

people_1778 = people_1778.drop(columns = ["Двори джерело", "Був на сповіді", "Не був", "Не був за малолітством", "Чоловіки", "Жінки", "Жінки наскрізне",
                                          "Чоловіки наскрізне", "Клас", "Категорія"])

people_1778[["Аркуш", "ID домогосподарство", "По-батькові", "Прізвище"]] = (
    people_1778[["Аркуш", "ID домогосподарство", "По-батькові", "Прізвище"]].ffill())

people_1778 = people_1778.rename(columns = {"ID наскрізна": "ID від джерела", "ID домогосподарство": "ID домогосподарства"})
people_1778["Рік джерела"] = pd.array([1778]*len(people_1778), dtype = int)

people_1778 = people_1778[["Рік джерела", "Аркуш", "ID домогосподарства", "ID від джерела", "Імʼя", "По-батькові", "Прізвище", "Вік", "Стать", "Родиний статус", "Соціальний статус"]]

people_1778.head()

,Рік джерела,Аркуш,ID домогосподарства,ID від джерела,Імʼя,По-батькові,Прізвище,Вік,Стать,Родиний статус,Соціальний статус
0,1778,445,1,1,Федор,Кондратьев,Лукашевич,44,Чол,господар,духовные
1,1778,445,1,2,Варвара,Симева,Лукашевич,35,Жін,жена,духовные
2,1778,445,1,3,Анна,Симева,Лукашевич,17,Жін,дочь,духовные
3,1778,445,1,4,Тимофей,Симева,Лукашевич,15,Чол,сын,духовные
4,1778,445,1,5,Пантелеймон,Симева,Лукашевич,14,Чол,сын,духовные


In [6]:
people_1897 = pd.read_csv("2.1_Дунаєць.xlsx - База людей.csv")

people_1897[["Імʼя", "По-батькові", "Прізвище"]] = people_1897["ФИО"].str.strip().str.replace("' ", "'").str.split(" ", expand = True)

people_1897 = people_1897.dropna(subset = ["ID строки в базе"])
people_1897 = people_1897.drop(columns = ["Link", "Кто заполнял базу", "Профессия вспомогательное", "Положение по воинской повинности", "Примітки", "ID жилец",
                                          "Здесь ли родился", "Место рождения", "Здесь ли приписан", "Здесь ли обыкновенно проживает", "Отметка об отсуствии",
                                          "Родной язык", "Умеет ли читать", "Обучение", "Семейный статус", "Вероисповедание", "Профессия главное", "ФИО"])
                                                # Знову повторюсь, для зєднання 2ох датасетів в один, я сильно ріжу дані
                                                # Далі можливо для індивідуальної роботи з ними я перероблю код та не буду видаляти так багато стовпців

people_1897["Рік джерела"] = pd.array([1897]*len(people_1897), dtype = int)
people_1897[["ID строки в базе", "ID Домохозяйствао"]] = people_1897[["ID строки в базе", "ID Домохозяйствао"]].astype(int)

def get_list_num(x):
    res = list(x.split("Арк."))
    return res[1]
people_1897["List ID"] = people_1897["List ID"].apply(get_list_num)
people_1897["Пол"] = people_1897["Пол"].str.replace("m", "Чол").str.replace("f", "Жін")

people_1897 = people_1897.rename(columns = {"ID строки в базе":"ID від джерела", "ID Домохозяйствао":"ID домогосподарства", "Возраст":"Вік",
                                            "Глава хозяйства и глава семьи":"Родиний статус", "Сословие, состояние или звание":"Соціальний статус", "Пол":"Стать",
                                            "List ID":"Аркуш"})

people_1897 = people_1897[["Рік джерела", "Аркуш", "ID домогосподарства", "ID від джерела", "Імʼя", "По-батькові", "Прізвище", "Вік", "Стать", "Родиний статус", "Соціальний статус"]]


people_1897.head()

,Рік джерела,Аркуш,ID домогосподарства,ID від джерела,Імʼя,По-батькові,Прізвище,Вік,Стать,Родиний статус,Соціальний статус
0,1897,1-2,1,1,Krasovskij,Pantelejmon,Ivanovich,32,Чол,husband,cossack
1,1897,1-2,1,2,Krasovskaya,Stefanida,Artemieva,28,Жін,wife,cossack
2,1897,1-2,1,3,Krasovskaya,Anna,Pantelejmonova,5,Жін,daughter,cossack
3,1897,1-2,1,4,Krasovskij,Stefan,Pantelejmonov,1,Чол,son,cossack
4,1897,1-2,1,5,Krasovskij,Ivan,Kondratov,70,Чол,father,cossack


In [7]:
people = pd.concat([people_1778, people_1897], ignore_index = True)

def better_age(x):
    if pd.isna(x):
        return 0
    x_str = str(x).lower()
    if "month" in x_str or "months" in x_str or "days" in x_str:
        return 0
    if str(x).replace(".", "", 1).isdigit():
        return int(float(x))
    return 0
people["Вік"] = people["Вік"].apply(better_age)

people.head()

,Рік джерела,Аркуш,ID домогосподарства,ID від джерела,Імʼя,По-батькові,Прізвище,Вік,Стать,Родиний статус,Соціальний статус
0,1778,445,1,1,Федор,Кондратьев,Лукашевич,44,Чол,господар,духовные
1,1778,445,1,2,Варвара,Симева,Лукашевич,35,Жін,жена,духовные
2,1778,445,1,3,Анна,Симева,Лукашевич,17,Жін,дочь,духовные
3,1778,445,1,4,Тимофей,Симева,Лукашевич,15,Чол,сын,духовные
4,1778,445,1,5,Пантелеймон,Симева,Лукашевич,14,Чол,сын,духовные


Тепер ми маємо зручний для роботи датасет, в якому поєднані дані як з 1778, так і з 1897 років.
Наразі в ньому є певні моменти, які можна було б покращити (на приклад можна транслітерувати певні колонки).
Але я не став цього робити бо намагався знайти компроміс між приведенням даних до зручного формату,
та збереженням їх оригінальності.
Тому в решті решт відмовився від деяких змін.

Такий датасет підійде для загального опрацювання даних.
Але, звісно, пізніше, якщо виникне потреба аналізувати більш глибокі питання я зможу легко переробити файл під конкретну тему, щоб зберігти потрібні дані які тут довелось видалити для "чистоти", та дослідити питання більш глибоко.


## Частина 2: Загальний аналіз


### Віковий розподіл населення

#### Особливості вікової структури населення, як змінилося за сто років

Нарешті маючи датасет можемо присиупити до загального аналізу даних.

Першим питанням, що ми розглянемо буде: "Особливості вікової структури населення, як змінилося за сто років".
Для цього побудую лінійний графік кількості населення по віковим групам, а також їх відсоткової частки від загальної кількості:


In [8]:
lineral_age = alt.Chart(people).mark_line().encode(
    x = alt.X("Вік", title = "Вік", bin = alt.Bin(maxbins = 20), axis = alt.Axis(grid = True)),
    y = alt.Y("count()", title = "Кількість людей"),
    color = alt.Color("Рік джерела:O", legend = alt.Legend(orient="top-right"))
).properties(
    title = "Кількість населення за віком",
    width = 500,
    height = 300)


people["percent"] = 0.0

people.loc[people["Рік джерела"] == 1778, "percent"] = 100 / len(people[people["Рік джерела"] == 1778])
people.loc[people["Рік джерела"] == 1897, "percent"] = 100 / len(people[people["Рік джерела"] == 1897])

percent_age = alt.Chart(people).mark_line().encode(
    x = alt.X("Вік", title = "Вік", bin = alt.Bin(maxbins = 20), axis = alt.Axis(grid = True)),
    y = alt.Y("sum(percent):Q", title = "Відсоток людей"),
    color = alt.Color("Рік джерела:O", legend = alt.Legend(orient = "top-right"))
).properties(
    title = "Відсоткове співвідношення населення за віком",
    width = 500,
    height = 300)


alt.vconcat(lineral_age, percent_age)

alt.VConcatChart(...)

В результаті бачимо доволі очікуваний спад, як кількості населення, так і його частки; разом зі збільшенням віку людей.
Найчисленнішою групою є діти віком до 5ти років, хоча майде відразу після 10ти років кількість людей падає.
Це пов'язано з високою дитячою смертністю і ця тенденція присутня з майже ідентичними показниками як в 1778ому так і в 1897ому.

Що до відмінностей між часовими періодами, варто відзначити групу 20-25 років,
в якій дані з 1778 мають помітно більші значення (Можливо це пов'язано з війною/репресіями/хворобами або іншими геополітичними обставинами).
А також чітко видно перевагу для вікових груп 60+ для яких смертність в рази скоротилась в 1897ому порівняно з 1778мим роком (Це скоріш всього викликано покращенням медицини та загальним рівнем життя в регіоні).

Ще один момент який можна помітити - це те, що графіки кількості та відсоткового спів відношення, майже не відрізняються.
Це пов'язано з кількістю населення, яке є майже ідентичним для обох часових періодів. Якби в одному з датасетів було сильно більше спостережень, то графік кількості був би незручним для порівняння і тоді графік співвідношеннь, був би більш корисним.


#### Статеве співвідношення різних вікових груп, як змінилося за сто років

Далі розглянемо "Статеве співвідношення різних вікових груп, як змінилося за сто років".
Для цього побудую статево-вікову піраміду для кожного з датасетів:

In [9]:
data_1778 = people[people["Рік джерела"] == 1778][["Вік", "Стать"]].copy()

men_1778 = alt.Chart(data_1778[data_1778["Стать"] == "Чол"]
).transform_aggregate(
    count = "count()",
    groupby = ["Вік"]        # Довго шукав як розвернути чоловіків щоб зробити піраміду, таки знайшов
).transform_calculate(
    neg_count = "-datum.count"
).mark_bar().encode(
    x = alt.X("neg_count:Q", title = "Кількість", scale = alt.Scale(domain = [-40, 40])),
    y = alt.Y("Вік:O", sort = "descending", axis = alt.Axis(grid = True, values = list(range(0, 86, 5)))),
    color = alt.value("#67a9cf"))

women_1778 = alt.Chart(data_1778[data_1778["Стать"] == "Жін"]
).transform_aggregate(
    count = "count()",
    groupby = ["Вік"]
).mark_bar().encode(
    x = alt.X("count:Q", title = "Кількість", scale = alt.Scale(domain = [-40, 40])),
    y = alt.Y("Вік:O", sort = "descending", axis = alt.Axis(grid = True, values = list(range(0, 86, 5)))),
    color = alt.value("#ef8a62"))

age_1778 = (men_1778 + women_1778).properties(title = "Віково-статева піраміда 1778", height = 500)



data_1897 = people[people["Рік джерела"] == 1897][["Вік", "Стать"]].copy()

men_1897 = alt.Chart(data_1897[data_1897["Стать"] == "Чол"]
).transform_aggregate(
    count = "count()",
    groupby = ["Вік"]
).transform_calculate(
    neg_count = "-datum.count"
).mark_bar().encode(
    x = alt.X("neg_count:Q", title = "Кількість", scale = alt.Scale(domain = [-40, 40])),
    y = alt.Y("Вік:O", sort = "descending", axis = alt.Axis(grid = True, values = list(range(0, 86, 5)))),
    color = alt.value("#67a9cf"))

women_1897 = alt.Chart(data_1897[data_1897["Стать"] == "Жін"]
).transform_aggregate(
    count = "count()",
    groupby = ["Вік"]
).mark_bar().encode(
    x = alt.X("count:Q", title = "Кількість", scale = alt.Scale(domain = [-40, 40])),
    y = alt.Y("Вік:O", sort = "descending", axis = alt.Axis(grid = True, values = list(range(0, 86, 5)))),
    color = alt.value("#ef8a62"))

age_1897 = (men_1897 + women_1897).properties(title = "Віково-статева піраміда 1897", height = 500)



piramid_age = alt.concat(age_1778, age_1897).resolve_scale(y = "shared")

piramid_age

alt.ConcatChart(...)

Бачимо в цілому деяку "рваність" даних, це пов'язано з маленькою вибіркою і трохи ускладнює аналіз, але певні тенденції розгледіти можна.

По-перше, помітні моменти які я вже зазначав вище - це велика кількість дітей до 5ти років з різким спадом в районі 10ти,
а також більше число людей віком 60+ в 1897ому порівняно з 1778мим роком.

Що до статевих відмінностей у 1778ому році бачимо певну невелику домінацію кількості чоловіків над жінками в молодому віці,
яка втім змінюється на домінацію жінок після позначки в 25ть років.
Натомість в 1897ому, ситуація дзеркальна: жінки трохи домінують в кількості у віці до 15ти років, а потім поступаються чоловікам.

Ще один незначний момент пов'язаний з датасетами, це відсутність у даних 1778го року дітей віком менше 12 місяців (тобто 0 років).
Це може бути пов'язано з способом перепису в цей період, таких дітей могли записувати як однорічних, або просто не вносити до реєстру через що в датасеті нема нульових значень віку.

### Господарства (краще назви не придумав)

#### Розміри домогосподарств; як це змінилось за сто років

Тут розглянемо домогосподарства, їх кількість та кількість людей що в них живуть. Для цього я побудую прості стовпцеві графіки для порівняння та графік розподілу:

In [10]:
data_hosp = people.groupby(["Рік джерела", "ID домогосподарства"]).size().reset_index(name = "Кількість людей в господарстві")

bar_hosp = alt.Chart(data_hosp).mark_bar().encode(
    x = alt.X("Рік джерела:O", title = "Рік",
        axis = alt.Axis(labelAngle = 0, labelFontSize = 12, titleFontSize = 14)),
    y = alt.Y("count()", title = "Кількість домогосподарств",
        axis=alt.Axis(titleFontSize = 14, labelFontSize = 12)),
    color = alt.Color("Рік джерела:O", legend=None)
).properties(
    title = "Кількість домогосподарств за роками",
    width = 200,
    height = 400)


rospodil_hosp = alt.Chart(data_hosp).mark_line(interpolate = "monotone").encode(
    x = alt.X("Кількість людей в господарстві", bin = alt.Bin(maxbins = 20), title = "Кількість людей у господарстві",
        axis = alt.Axis(labelFontSize = 12, titleFontSize = 14)),
    y = alt.Y("count()", title = "Кількість домогосподарств",
        axis = alt.Axis(labelFontSize = 12, titleFontSize = 14)),
    color = alt.Color("Рік джерела:O", title = "Рік джерела", legend = alt.Legend(orient = "top-right", labelFontSize = 14, titleFontSize = 14)),
).properties(
    title = "Розподіл кількості людей в домогосподарстві",
    width = 600,
    height = 400)


data_hosp = data_hosp.groupby("Рік джерела")["Кількість людей в господарстві"].mean().reset_index()

bar_people_in_hosp = alt.Chart(data_hosp).mark_bar().encode(
    x = alt.X("Рік джерела:O", title = "Рік",
        axis = alt.Axis(labelAngle = 0, labelFontSize = 12, titleFontSize = 14)),
    y = alt.Y("Кількість людей в господарстві", title = "Середня кількість людей",
        axis=alt.Axis(titleFontSize = 14, labelFontSize = 12)),
    color = alt.Color("Рік джерела:O", legend = None)
).properties(
    title = "Середня кількість людей в домогосподарстві",
    width = 200,
    height = 400)

alt.concat(bar_hosp, rospodil_hosp, bar_people_in_hosp).resolve_scale(color = "independent")


alt.ConcatChart(...)

Бачимо очевидні відмінності між овома часовими проміжками. У 1897ому році кількість господарств зрасла більш ніж вдвічі порівняно з 1778мим роком.
Одночасно середня кількість людей в домогосподарстві пропорційно скоротилася.
З цього можна зробити висновок, що за більш ніж 100 років, люди пристосувались до нового способу співіснування та розподілю праці.
Підозрюю, що раніше сільське господарство, та отримання їжу вимагало більших зусиль і робочих рук. Тому люди і об'єднувались у великі групи, щоб розподілити обов'язки.
Але з плином часу та технологічним прогресом, робота більше не вимагала великих зусиль та об'єднання в численні групи.
Тому люди все частіше будували господарства з невеликою численністю населення. Як наслідок і кількість господарств зросла, хоч тепер вони і не були такими великими як раніше.

Що ж до середнього графіку розподілу: бачимо вже проговорені тенденції. В 1897ому більшість господарств має від 5ти до 10 осіб.
В той час як у 1778ому основна кількість зконцентрована в районі 10ти - 20ти осіб на господарство, також бачимо довгий хвіст у даних, який доходить до 70+ людей на господарство.
Варто підмітити, що цей хвіст скоріш всього вплинув на таку значну відміну в середніх значеннях, і при його відсутності дані між роками, були б не настільки відмінними.
Можемо зробити висновок, що хоч тенденція на спад присутня, вона не була б такою виразною, якби не прорівняно невелика кількість густонаселених домогосподарств у 1778ому році

### Справи сімейні

#### Структура родин; як це змінилось за сто років

Останньою темою, яку розглянемо буде "Структура родин; як це змінилось за сто років". Дли цього використаю 2 датасети "Структура родини" за 1778ий та за 1897ий роки.
Вони вже довогі гарно зведені, але я все одно трохи перероблю та з'єднаю їх для зручності:

In [11]:
family_1778 = pd.read_csv("2.2.2_Структура родини Дунаєць_1778.xlsx - Total.csv")
family_1897 = pd.read_csv("2.1.1_Структура родини Дунаєць_1897.xlsx - Total.csv")

family_1778[["Категорія", "%"]] = family_1778[["Категорія", "%"]].ffill()
family_1897[["Категорія", "%"]] = family_1897[["Категорія", "%"]].ffill()

family_1778["Рік джерела"] = 1778
family_1897["Рік джерела"] = 1897

family = pd.concat([family_1778, family_1897], ignore_index = True)

family = family.dropna(subset = ["Клас"])

family.head()

,Категорія,Клас,Кількість,%,% кожного класу,Усього по категоріях,Рік джерела
0,Самотні особи,1a Вдови і вдівці,0,0.0,0.0,NaN,1778
1,Самотні особи,"1b Нежонаті, або неокресленого цивільного стану",0,0.0,0.0,0.0,1778
2,Безструктурні,"2a Нежонаті брати і сестри, які живуть разом",0,0.0,0.0,NaN,1778
3,Безструктурні,"2b Інші кревні, які живуть разом",0,0.0,0.0,0.0,1778
4,Нуклеарні,3a Шлюбі пари без дітей,1,18.9,0.9,NaN,1778


Тепер порівняємо Часові періоди, за категоріями. Для цього я б хотів використати барчарт, але їх і так багато в цьому проекті, тому пайчарт:

In [12]:
pie_fam = family[["Категорія", "Усього по категоріях", "Рік джерела"]].copy().dropna(subset = ["Усього по категоріях"])

pie_1778 = alt.Chart(pie_fam[pie_fam["Рік джерела"] == 1778]).mark_arc().encode(
    theta = alt.Theta(field="Усього по категоріях", type = "quantitative"), # тут чомусь саме в пайчатні тип даних треба прописувати через type =
    color = alt.Color(field="Категорія", type = "nominal", legend = alt.Legend(title = "Категорія")), # коли я писав :Q у мене непрацювали кольори
    tooltip = ["Категорія", "Усього по категоріях", "Рік джерела"]
).properties(
    title = "1778 рік",
    width = 250,
    height = 250)

pie_1897 = alt.Chart(pie_fam[pie_fam["Рік джерела"] == 1897]).mark_arc().encode(
    theta = alt.Theta(field = "Усього по категоріях", type = "quantitative"),
    color = alt.Color(field = "Категорія", type = "nominal", legend = alt.Legend(title = "Категорія")),
    tooltip = ["Категорія", "Усього по категоріях", "Рік джерела"]
).properties(
    title = "1897 рік",
    width = 250,
    height = 250)

alt.concat(pie_1778, pie_1897).properties(title = alt.TitleParams(
    text = "Відношення типів родин в різні роки",
    anchor = "middle",
    fontSize = 16))

alt.ConcatChart(...)

Бачимо, що пропорції різних типів помінто змінились за 100 років, давайте по порядку:
- Безструктурні займають нульову частку і 1778ому та дуже малу в 1897ому, тому про них сказати нема чого.
- Мультифокальні (сім'ї без чітко вираженого(ї) голови сімейства) займають більше 75% у 1778ому, але з плином часу їх частка зменшується і у 1897ому їх частка становить приблизно третину. Це, до речі, повязано з минулим графіком де ми побачили, як великі господарства зникали і розбивались на менші, так само і кількість великих розпорошених сімей стала меншою.
- Нуклеарні (сім'ї з чітко вираженою головою сімейства) натомість збільшились більш ніж вдвічі порівняно з 1778им роком, і у 1897ому вони займають вже більше третини. Тобто менших сімей де є лише 1 голова стало більше.
- Розширені (Нуклеарні сім'ї + додаткові родичі типу бабусі\дідуся) сильно зросли від майже непомітної частинки у 1778ому, до приблизно 20% від загальної кількості у 1897ому. Зновуж підмічу як кількість менших сімей збільшується а великих зменшується і як це корелює з аналогічною тенденцією у господарствах.
- Самотні особи, були повністю відсутні у 1778ому і з'явились у невеликій кількості у 1897ому. Це зайвий раз підтверджує, що якість життя зросла, адже життя самотушки, яке раніше було неможливим, тепер стало реальним через легші умови.

#### Як змінився класовий розподіл з часом

І останнім питанням, що ми розглянемо буде "Як змінився класовий розподіл з часом". Для цього я побудую простенький лінійний графік з лише 2ма точками на вісі x.
Так ми зможемо відразу побачити тенденцію на зріст чи спад кожного з класів:

In [13]:
line_chart = alt.Chart(family).mark_line(point=True).encode(
    x = alt.X("Рік джерела:Q", title = "Рік", axis = alt.Axis(titleFontSize = 16, labelFontSize = 14)),
    y = alt.Y("% кожного класу", title = "Відсоток кожного класу", axis = alt.Axis(titleFontSize = 16, labelFontSize = 14)),
    color = alt.Color("Клас", title = "Клас родини", scale = alt.Scale(scheme = "category20"),
                      legend = alt.Legend(orient = "top-right",titleFontSize = 16, labelFontSize = 14, labelLimit = 200))
).properties(
    width = 400,
    height = 700,
    title = alt.TitleParams(
        text = "Динаміка частки кожного класу між 1778 і 1897 роками",
        anchor = "middle",
        fontSize = 14))

line_chart

alt.Chart(...)

Я не буду коментувати кожен з класів родин, адже вони по суті є підмножинами категорій родин і повторюють їх основні тенденції.
Відзначу лише, 3 класи, що виділяються. Навідміну від інших класів, що зросли, ці 3 зазнали спаду:
- 5e інші, тут нема чого сказати, скоріш всього в датасеті 1778го року були погано класифіковані родини, тому велика частка була просто позначена як інші. Це скоріш всього трохи вплинуло на чистоту даних, але певні висновки, все одно зробити можна.
- 5d братські родини, спали в кількості вроїдно це пов'язано з розпадом родин на менші, про який я вже згадував.
- 5c родини з бічними другорядними ядрами, тобто великі родини в яких немає чітковираженого лідера, знову ж їх кількість спадає.

## Частина 3: Фінал

#### Коротко підведу висновки:

Оптимізація даних, зведення їх до зручного вигляду та подальший аналіз показав кілька цікавих тенденцій, що пов'язані між собою та підтверджують одна одну.
Основними аспектоми, які вдалось помітити та підтвердити, є покращення рівня життя людей і, як наслідок, зміна стилю їхнього життя до більш самостійного та зручного для них самих.
Це підтверджується як даними про статево-вікові співвідношення, так і зміною структури господарств та сімей.

Фух, дякую за увагу, зара ще запишу відос і кидаю вам цей шедевр.